<a href="https://colab.research.google.com/github/AshokGit544/Model-Ready-Utility-Datasets-Project/blob/main/Model_Ready_Utility_Datasets_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from pathlib import Path
from datetime import datetime, timedelta
import random
import json

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, classification_report

BASE_PATH = Path('/content/utility_model_ready_dataset_preparation')
RAW_PATH = BASE_PATH / 'data' / 'raw'
PROCESSED_PATH = BASE_PATH / 'data' / 'processed'
MODEL_READY_PATH = BASE_PATH / 'data' / 'model_ready'
OUTPUT_PATH = BASE_PATH / 'outputs'
CONFIG_PATH = BASE_PATH / 'config'

for p in [RAW_PATH, PROCESSED_PATH, MODEL_READY_PATH, OUTPUT_PATH, CONFIG_PATH]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(42)
np.random.seed(42)

num_customers = 2000
num_meter_records = 30000
num_outages = 2500
num_operational = 12000
num_weather = 1200
regions = ['North', 'South', 'East', 'West', 'Central']
start_date = datetime(2024, 1, 1)

print("Setup completed successfully.")
print(BASE_PATH)

Setup completed successfully.
/content/utility_model_ready_dataset_preparation


In [4]:
# customer region mapping
customer_master = []
for i in range(1, num_customers + 1):
    customer_master.append({
        'customer_id': f'CUST{i:05d}',
        'region': random.choice(regions),
        'customer_type': random.choice(['Residential', 'Commercial', 'Industrial'])
    })

customer_master_df = pd.DataFrame(customer_master)
customer_master_df.to_csv(RAW_PATH / 'customer_master.csv', index=False)

# smart meter data
meter = []
for i in range(1, num_meter_records + 1):
    cust = random.randint(1, num_customers)
    read_date = start_date + timedelta(days=random.randint(0, 179))
    meter.append({
        'meter_record_id': f'MTR{i:07d}',
        'customer_id': f'CUST{cust:05d}',
        'reading_date': read_date.strftime('%Y-%m-%d'),
        'kwh_usage': round(float(max(5, np.random.normal(35, 12))), 2),
        'estimated_read_flag': random.choice([0, 0, 0, 1])
    })

meter_df = pd.DataFrame(meter)
meter_df.to_csv(RAW_PATH / 'meter_usage.csv', index=False)

# outage data
outages = []
for i in range(1, num_outages + 1):
    cust = random.randint(1, num_customers)
    outage_date = start_date + timedelta(days=random.randint(0, 179))
    outages.append({
        'outage_id': f'OUT{i:06d}',
        'customer_id': f'CUST{cust:05d}',
        'outage_date': outage_date.strftime('%Y-%m-%d'),
        'duration_minutes': max(5, int(np.random.normal(90, 35))),
        'cause': random.choice(['Storm', 'Equipment Failure', 'Vegetation', 'Animal Contact', 'Unknown'])
    })

outages_df = pd.DataFrame(outages)
outages_df.to_csv(RAW_PATH / 'outages.csv', index=False)

# operational data
operational = []
for i in range(1, num_operational + 1):
    cust = random.randint(1, num_customers)
    op_date = start_date + timedelta(days=random.randint(0, 179))
    operational.append({
        'operation_id': f'OPS{i:07d}',
        'customer_id': f'CUST{cust:05d}',
        'event_date': op_date.strftime('%Y-%m-%d'),
        'load_factor': round(float(max(0.1, np.random.normal(0.72, 0.15))), 3),
        'voltage_variation': round(float(abs(np.random.normal(2.5, 1.2))), 2)
    })

operational_df = pd.DataFrame(operational)
operational_df.to_csv(RAW_PATH / 'operational_metrics.csv', index=False)

# weather data
weather = []
for i in range(num_weather):
    weather_date = start_date + timedelta(days=i % 180)
    weather.append({
        'weather_date': weather_date.strftime('%Y-%m-%d'),
        'region': random.choice(regions),
        'temperature_f': round(float(np.random.normal(62, 18)), 2),
        'rainfall_mm': round(float(max(0, np.random.normal(4, 5))), 2),
        'wind_speed_mph': round(float(max(0, np.random.normal(14, 6))), 2),
        'storm_flag': random.choice([0, 0, 0, 1])
    })

weather_df = pd.DataFrame(weather)
weather_df.to_csv(RAW_PATH / 'weather.csv', index=False)

print('Raw datasets created successfully.')
for file in RAW_PATH.glob('*.csv'):
    print(file.name)

Raw datasets created successfully.
operational_metrics.csv
weather.csv
outages.csv
customer_master.csv
meter_usage.csv


In [5]:
feature_dictionary = {
    'target_use_cases': ['forecasting', 'anomaly_analysis', 'grid_reliability'],
    'key_features': [
        'daily_kwh_usage',
        'usage_7day_avg',
        'usage_30day_avg',
        'daily_outage_count',
        'daily_outage_minutes',
        'load_factor_avg',
        'voltage_variation_avg',
        'temperature_f',
        'rainfall_mm',
        'wind_speed_mph',
        'storm_flag'
    ],
    'business_keys': ['customer_id', 'event_date', 'region']
}

with open(CONFIG_PATH / 'feature_dictionary.json', 'w') as f:
    json.dump(feature_dictionary, f, indent=2)

print(json.dumps(feature_dictionary, indent=2))

{
  "target_use_cases": [
    "forecasting",
    "anomaly_analysis",
    "grid_reliability"
  ],
  "key_features": [
    "daily_kwh_usage",
    "usage_7day_avg",
    "usage_30day_avg",
    "daily_outage_count",
    "daily_outage_minutes",
    "load_factor_avg",
    "voltage_variation_avg",
    "temperature_f",
    "rainfall_mm",
    "wind_speed_mph",
    "storm_flag"
  ],
  "business_keys": [
    "customer_id",
    "event_date",
    "region"
  ]
}


In [6]:
customer_master_df = pd.read_csv(RAW_PATH / 'customer_master.csv')
meter_df = pd.read_csv(RAW_PATH / 'meter_usage.csv')
outages_df = pd.read_csv(RAW_PATH / 'outages.csv')
operational_df = pd.read_csv(RAW_PATH / 'operational_metrics.csv')
weather_df = pd.read_csv(RAW_PATH / 'weather.csv')

meter_df['reading_date'] = pd.to_datetime(meter_df['reading_date'])
outages_df['outage_date'] = pd.to_datetime(outages_df['outage_date'])
operational_df['event_date'] = pd.to_datetime(operational_df['event_date'])
weather_df['weather_date'] = pd.to_datetime(weather_df['weather_date'])

meter_df = meter_df.drop_duplicates(subset=['customer_id', 'reading_date'])
outages_df = outages_df[outages_df['duration_minutes'] >= 0]
operational_df = operational_df[(operational_df['load_factor'] >= 0) & (operational_df['voltage_variation'] >= 0)]

customer_master_df.to_csv(PROCESSED_PATH / 'customer_master_processed.csv', index=False)
meter_df.to_csv(PROCESSED_PATH / 'meter_usage_processed.csv', index=False)
outages_df.to_csv(PROCESSED_PATH / 'outages_processed.csv', index=False)
operational_df.to_csv(PROCESSED_PATH / 'operational_metrics_processed.csv', index=False)
weather_df.to_csv(PROCESSED_PATH / 'weather_processed.csv', index=False)

print('Processed datasets saved successfully.')

Processed datasets saved successfully.


In [7]:
meter_daily = meter_df.groupby(['customer_id', 'reading_date'], as_index=False).agg(
    daily_kwh_usage=('kwh_usage', 'sum'),
    has_estimated_read=('estimated_read_flag', 'max')
).rename(columns={'reading_date': 'event_date'})

outage_daily = outages_df.groupby(['customer_id', 'outage_date'], as_index=False).agg(
    daily_outage_count=('outage_id', 'count'),
    daily_outage_minutes=('duration_minutes', 'sum')
).rename(columns={'outage_date': 'event_date'})

operational_daily = operational_df.groupby(['customer_id', 'event_date'], as_index=False).agg(
    load_factor_avg=('load_factor', 'mean'),
    voltage_variation_avg=('voltage_variation', 'mean')
)

base_df = meter_daily.merge(customer_master_df, on='customer_id', how='left')
base_df = base_df.merge(outage_daily, on=['customer_id', 'event_date'], how='left')
base_df = base_df.merge(operational_daily, on=['customer_id', 'event_date'], how='left')

weather_daily = weather_df.rename(columns={'weather_date': 'event_date'})
base_df = base_df.merge(weather_daily, on=['event_date', 'region'], how='left')

base_df['daily_outage_count'] = base_df['daily_outage_count'].fillna(0)
base_df['daily_outage_minutes'] = base_df['daily_outage_minutes'].fillna(0)
base_df['load_factor_avg'] = base_df['load_factor_avg'].fillna(base_df['load_factor_avg'].median())
base_df['voltage_variation_avg'] = base_df['voltage_variation_avg'].fillna(base_df['voltage_variation_avg'].median())
base_df['temperature_f'] = base_df['temperature_f'].fillna(base_df['temperature_f'].median())
base_df['rainfall_mm'] = base_df['rainfall_mm'].fillna(0)
base_df['wind_speed_mph'] = base_df['wind_speed_mph'].fillna(base_df['wind_speed_mph'].median())
base_df['storm_flag'] = base_df['storm_flag'].fillna(0)

base_df = base_df.sort_values(['customer_id', 'event_date'])

base_df['usage_7day_avg'] = base_df.groupby('customer_id')['daily_kwh_usage'].transform(lambda x: x.rolling(7, min_periods=1).mean())
base_df['usage_30day_avg'] = base_df.groupby('customer_id')['daily_kwh_usage'].transform(lambda x: x.rolling(30, min_periods=1).mean())
base_df['prev_day_usage'] = base_df.groupby('customer_id')['daily_kwh_usage'].shift(1).fillna(0)
base_df['outage_risk_flag'] = np.where(base_df['daily_outage_minutes'] > 120, 1, 0)
base_df['usage_anomaly_flag'] = np.where(base_df['daily_kwh_usage'] > (base_df['usage_30day_avg'] * 1.5), 1, 0)
base_df['month'] = pd.to_datetime(base_df['event_date']).dt.month
base_df['day_of_week'] = pd.to_datetime(base_df['event_date']).dt.dayofweek

base_df.to_csv(MODEL_READY_PATH / 'utility_model_ready_dataset.csv', index=False)
print('Model-ready dataset created successfully.')
print(base_df.head())

Model-ready dataset created successfully.
  customer_id event_date  daily_kwh_usage  has_estimated_read region  \
0   CUST00001 2024-01-01            27.10                   0  North   
1   CUST00001 2024-01-11            31.12                   0  North   
2   CUST00001 2024-01-20            40.52                   0  North   
3   CUST00001 2024-03-03            42.74                   0  North   
4   CUST00001 2024-04-07            27.66                   0  North   

  customer_type  daily_outage_count  daily_outage_minutes  load_factor_avg  \
0   Residential                 0.0                   0.0           0.7215   
1   Residential                 0.0                   0.0           0.7215   
2   Residential                 0.0                   0.0           0.7215   
3   Residential                 0.0                   0.0           0.7215   
4   Residential                 0.0                   0.0           0.7215   

   voltage_variation_avg  ...  rainfall_mm  wind_speed_m

In [8]:
regional_kpis = base_df.groupby('region', as_index=False).agg(
    avg_daily_kwh_usage=('daily_kwh_usage', 'mean'),
    avg_daily_outage_minutes=('daily_outage_minutes', 'mean'),
    avg_load_factor=('load_factor_avg', 'mean'),
    outage_risk_rate=('outage_risk_flag', 'mean')
)

regional_kpis.to_csv(OUTPUT_PATH / 'regional_feature_kpis.csv', index=False)
regional_kpis.head()

,region,avg_daily_kwh_usage,avg_daily_outage_minutes,avg_load_factor,outage_risk_rate
0,Central,34.998509,0.520968,0.721801,0.000940
1,East,35.049008,0.487011,0.722292,0.000431
2,North,34.634972,0.455513,0.720984,0.001121
3,South,35.002100,0.886756,0.721273,0.002559
4,West,34.995291,0.746398,0.721026,0.001210


In [9]:
forecast_df = base_df.copy()
forecast_df = forecast_df.dropna(subset=['daily_kwh_usage'])

features_reg = [
    'usage_7day_avg', 'usage_30day_avg', 'prev_day_usage',
    'daily_outage_minutes', 'load_factor_avg', 'voltage_variation_avg',
    'temperature_f', 'rainfall_mm', 'wind_speed_mph', 'storm_flag',
    'month', 'day_of_week'
]

X_reg = forecast_df[features_reg]
y_reg = forecast_df['daily_kwh_usage']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg_model = RandomForestRegressor(n_estimators=100, random_state=42)
reg_model.fit(X_train_reg, y_train_reg)
preds_reg = reg_model.predict(X_test_reg)

mae = mean_absolute_error(y_test_reg, preds_reg)
print('Forecasting MAE:', round(mae, 2))

Forecasting MAE: 7.41


In [10]:
features_cls = [
    'daily_kwh_usage', 'usage_7day_avg', 'usage_30day_avg', 'prev_day_usage',
    'load_factor_avg', 'voltage_variation_avg', 'temperature_f',
    'rainfall_mm', 'wind_speed_mph', 'storm_flag', 'month', 'day_of_week'
]

X_cls = base_df[features_cls]
y_cls = base_df['outage_risk_flag']

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

cls_model = RandomForestClassifier(n_estimators=100, random_state=42)
cls_model.fit(X_train_cls, y_train_cls)
preds_cls = cls_model.predict(X_test_cls)

print(classification_report(y_test_cls, preds_cls))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9025
           1       0.00      0.00      0.00        12

    accuracy                           1.00      9037
   macro avg       0.50      0.50      0.50      9037
weighted avg       1.00      1.00      1.00      9037



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
feature_importance_df = pd.DataFrame({
    'feature': features_reg,
    'importance': reg_model.feature_importances_
}).sort_values('importance', ascending=False)

feature_importance_df.to_csv(OUTPUT_PATH / 'feature_importance_sample.csv', index=False)
feature_importance_df.head()

,feature,importance
0,usage_7day_avg,0.370248
2,prev_day_usage,0.177324
1,usage_30day_avg,0.117863
6,temperature_f,0.088402
8,wind_speed_mph,0.082888


In [12]:
summary = {
    'generated_at': datetime.now().isoformat(),
    'raw_files': len(list(RAW_PATH.glob('*.csv'))),
    'processed_files': len(list(PROCESSED_PATH.glob('*.csv'))),
    'model_ready_files': len(list(MODEL_READY_PATH.glob('*.csv'))),
    'model_ready_rows': len(base_df),
    'forecasting_mae': round(float(mae), 2)
}

with open(OUTPUT_PATH / 'model_ready_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "generated_at": "2026-03-17T05:41:32.698805",
  "raw_files": 5,
  "processed_files": 5,
  "model_ready_files": 1,
  "model_ready_rows": 45185,
  "forecasting_mae": 7.41
}
